In [28]:
# Localization_Regression.ipynb
import os
import numpy as np
import pandas as pd
import pydicom
import matplotlib.pyplot as plt
import vision_models.constants as constants
from pydicom.pixel_data_handlers.util import apply_modality_lut
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from keras import layers, models, applications, losses

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import tensorflow as tf
from keras import layers, models, applications, losses
from sklearn.metrics import confusion_matrix, classification_report

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report


import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import tensorflow as tf
from keras import layers, models, applications, losses
from sklearn.metrics import mean_squared_error, mean_absolute_error

import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import tensorflow as tf
from keras import layers, models, applications, losses
from sklearn.metrics import mean_squared_error, mean_absolute_error



In [29]:
# Set paths
train_images_path = constants.TRAIN_DATA_PATH
train_df = pd.read_csv("NEW_EDA/train_dataset.csv")
val_df = pd.read_csv("NEW_EDA/val_dataset.csv")
test_df = pd.read_csv("NEW_EDA/test_dataset.csv")

# # One-hot encode the 'condition' and 'level' columns separately
# train_condition_dummies = pd.get_dummies(train_df['condition'], prefix='condition')
# train_level_dummies = pd.get_dummies(train_df['level'], prefix='level')

# val_condition_dummies = pd.get_dummies(val_df['condition'], prefix='condition')
# val_level_dummies = pd.get_dummies(val_df['level'], prefix='level')

# test_condition_dummies = pd.get_dummies(test_df['condition'], prefix='condition')
# test_level_dummies = pd.get_dummies(test_df['level'], prefix='level')

# # Concatenate the dummy columns back to the original DataFrame
# train_df = pd.concat([train_df.drop(['condition', 'level'], axis=1), train_condition_dummies, train_level_dummies], axis=1)
# val_df = pd.concat([val_df.drop(['condition', 'level'], axis=1), val_condition_dummies, val_level_dummies], axis=1)
# test_df = pd.concat([test_df.drop(['condition', 'level'], axis=1), test_condition_dummies, test_level_dummies], axis=1)

# Now you have your DataFrame with one-hot encoded columns for 'condition' and 'level'


In [30]:
train_df = train_df.sample(frac=0.0005, random_state=42)
val_df = val_df.sample(frac=0.0005, random_state=42)
test_df = test_df.sample(frac=0.0005, random_state=42)

train_df

,study_id,series_id,image_name,file_meta_version,sop_class_uid,sop_instance_uid,transfer_syntax_uid,implementation_class_uid,implementation_version_name,content_date,...,bits_allocated,bits_stored,high_bit,pixel_representation,window_center,window_width,condition,level,x,y
29239,1820866003,3689221841,21.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,1820866003.1.21,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,16,12,11,0,401.0,885.0,Left Subarticular Stenosis,L5/S1,201.320251,205.563579
10576,3084269121,1616142642,3.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,3084269121.1.3,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,16,16,15,1,236.0,472.0,NaN,NaN,NaN,NaN
38534,4173917544,111096887,13.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,4173917544.1.13,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,16,12,11,0,481.0,1012.0,NaN,NaN,NaN,NaN
40490,2905685162,1535449979,14.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,2905685162.1.14,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,16,12,11,0,738.0,1534.0,NaN,NaN,NaN,NaN
115633,861719444,3451308655,87.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,861719444.1.87,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,16,16,15,1,730.0,1460.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49099,1302048123,272639520,38.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,1302048123.1.38,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,16,12,11,0,614.0,1270.0,Left Subarticular Stenosis,L4/L5,169.079025,161.028578
131137,3745670967,615061872,44.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,3745670967.1.44,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,16,12,11,0,686.0,1408.0,NaN,NaN,NaN,NaN
6818,1780646606,3098929855,21.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,1780646606.1.21,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,16,16,15,1,970.0,1941.0,Right Subarticular Stenosis,L1/L2,213.861947,250.336283
39227,3647070644,1825894086,23.dcm,b'\x00\x01',1.2.840.10008.5.1.4.1.1.4.1,3647070644.1.23,1.2.840.10008.1.2.5,1.2.40.0.13.1.1.1,PYDICOM 2.4.2,20240503,...,16,16,15,1,923.0,1847.0,NaN,NaN,NaN,NaN


In [31]:


# Data Preparation
for df in [train_df, val_df, test_df]:
    df['image_path'] = df.apply(lambda row: os.path.join(train_images_path, str(row['study_id']), str(row['series_id']), f"{row['instance_number']}.dcm"), axis=1)


In [32]:
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(69, 39)
(8, 39)
(9, 39)


In [33]:
# Keep only the necessary columns for the regression task
columns_to_keep = ['image_path', 'x', 'y']

# Drop all other columns
train_df = train_df[columns_to_keep]
val_df = val_df[columns_to_keep]
test_df = test_df[columns_to_keep]

print(train_df.columns)  # Check the remaining columns to ensure correctness
print(val_df.columns)
print(test_df.columns)


Index(['image_path', 'x', 'y'], dtype='object')
Index(['image_path', 'x', 'y'], dtype='object')
Index(['image_path', 'x', 'y'], dtype='object')


In [34]:

# Function to read and preprocess images using pydicom
def load_image(img_path, target_size=(224, 224)):
    # Load DICOM file
    dicom = pydicom.dcmread(img_path.numpy().decode('utf-8'))
    img = dicom.pixel_array
    
    # Normalize the image
    img = img / np.max(img)
    
    # Ensure img has 3 dimensions (Height, Width, Channels)
    if len(img.shape) == 2:  # Grayscale image
        img = np.expand_dims(img, axis=-1)  # Add channel dimension
    
    # Convert grayscale to RGB (3 channels) if needed
    if img.shape[-1] == 1:
        img = np.concatenate([img, img, img], axis=-1)
    
    # Convert to float32 immediately
    img = tf.convert_to_tensor(img, dtype=tf.float32)
    
    # Now resize the image
    img = tf.image.resize(img, target_size)
    
    # Normalize pixel values to [0, 1]
    img = img / 255.0
    
    return img

# Wrapper function to use with tf.data.Dataset.map
def load_image_wrapper(img_path, target_size=(224, 224)):
    img = tf.py_function(func=load_image, inp=[img_path], Tout=tf.float32)
    img.set_shape((target_size[0], target_size[1], 3))  # Explicitly set shape (224, 224, 3)
    return img

# Create a TensorFlow dataset for the regression task
def create_tf_dataset(df, batch_size=32, is_training=True):
    # Labels are the x and y coordinates for the regression task
    label_columns = ['x', 'y']
    
    dataset = tf.data.Dataset.from_tensor_slices((df['image_path'], df[label_columns]))
    dataset = dataset.map(lambda x, y: (load_image_wrapper(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    
    # Debugging: Print shape of each element in the dataset
    def print_shapes(x, y):
        print(f"Image shape: {x.shape}, Label shape: {y.shape}")
        return x, y
    
    dataset = dataset.map(print_shapes)  # Add this line to print shapes
    
    if is_training:
        dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size).prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset

# Example of creating datasets
train_dataset = create_tf_dataset(train_df)
val_dataset = create_tf_dataset(val_df, is_training=False)
test_dataset = create_tf_dataset(test_df, is_training=False)



Image shape: (224, 224, 3), Label shape: (2,)
Image shape: (224, 224, 3), Label shape: (2,)
Image shape: (224, 224, 3), Label shape: (2,)


In [35]:


# Ensure the directory exists
def ensure_directory(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

# Plot training history
def plot_metrics(history, title):
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{title} Loss')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend()
    save_path = f'NEW_EDA/LocalReg_{title}.png'
    ensure_directory(os.path.dirname(save_path))  # Ensure the directory exists
    plt.savefig(save_path)
    plt.close()  # Close the plot to ensure it's saved
    print(f"Training history saved at {save_path}")

# Function to save predictions and actual labels to CSV
def save_predictions_to_csv(true_coords, predicted_coords, df, dataset_name, model_type):
    # Create a DataFrame with true and predicted coordinates
    results_df = pd.DataFrame({
        'image_path': df['image_path'],
        'true_x': true_coords[:, 0],
        'true_y': true_coords[:, 1],
        'predicted_x': predicted_coords[:, 0],
        'predicted_y': predicted_coords[:, 1]
    })
    
    # Save to CSV
    save_path = f'NEW_EDA/LocalReg_{dataset_name}_{model_type}_predictions.csv'
    ensure_directory(os.path.dirname(save_path))  # Ensure the directory exists
    results_df.to_csv(save_path, index=False)
    print(f"LocalReg_{dataset_name}_{model_type}_predictions.csv saved successfully!")

# Evaluate the model and plot the results
def evaluate_model(model, dataset, df, dataset_name, model_type):
    # Get the true coordinates from the dataset
    true_coords = df[['x', 'y']].values
    
    # Make predictions
    predicted_coords = model.predict(dataset)
    
    # Calculate error metrics
    mae = mean_absolute_error(true_coords, predicted_coords)
    rmse = np.sqrt(mean_squared_error(true_coords, predicted_coords))
    
    print(f"Evaluation on {dataset_name} Data:")
    print(f"MAE: {mae}")
    print(f"RMSE: {rmse}")
    
    # Save predictions and actual coordinates to CSV
    save_predictions_to_csv(true_coords, predicted_coords, df, dataset_name, model_type)

# Function to create the CNN model for regression
def create_cnn_regression_model():
    model = models.Sequential([
        layers.Input(shape=(224, 224, 3)),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(2, activation='linear')  # Output layer with 2 neurons for x and y
    ])
    return model

# Function to create the ResNet model for regression
def create_resnet_regression_model():
    base_model = applications.ResNet50(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    base_model.trainable = False
    
    x = layers.GlobalAveragePooling2D()(base_model.output)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dense(64, activation='relu')(x)
    output = layers.Dense(2, activation='linear')(x)  # Output layer with 2 neurons for x and y
    
    model = models.Model(inputs=base_model.input, outputs=output)
    return model

# Function to dynamically create and train a model
def run_regression_model(model_type, train_df, val_df, test_df):
    if model_type == 'cnn':
        model = create_cnn_regression_model()
    elif model_type == 'resnet':
        model = create_resnet_regression_model()

    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    train_dataset = create_tf_dataset(train_df)
    val_dataset = create_tf_dataset(val_df, is_training=False)
    test_dataset = create_tf_dataset(test_df, is_training=False)

    history = model.fit(train_dataset, validation_data=val_dataset, epochs=10)
    
    title_suffix = f"{model_type.upper()} Model - Regression"
    plot_metrics(history, f"Training History for {title_suffix}")

    # Evaluate and save predictions for validation and test datasets
    evaluate_model(model, val_dataset, val_df, "Validation", model_type)
    evaluate_model(model, test_dataset, test_df, "Test", model_type)


In [36]:

# Example usage:
run_regression_model('cnn', train_df, val_df, test_df)  # Predict coordinates using CNN
run_regression_model('resnet', train_df, val_df, test_df)  # Predict coordinates using ResNet


2024-08-14 03:58:01.139711: W external/local_tsl/tsl/framework/bfc_allocator.cc:487] Allocator (GPU_0_bfc) ran out of memory trying to allocate 72.0KiB (rounded to 73728)requested by op AddV2
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2024-08-14 03:58:01.139750: I external/local_tsl/tsl/framework/bfc_allocator.cc:1044] BFCAllocator dump for GPU_0_bfc
2024-08-14 03:58:01.139761: I external/local_tsl/tsl/framework/bfc_allocator.cc:1051] Bin (256): 	Total Chunks: 50, Chunks in use: 50. 12.5KiB allocated for chunks. 12.5KiB in use in bin. 3.5KiB client-requested in use in bin.
2024-08-14 03:58:01.139768: I external/local_tsl/tsl/framework/bfc_allocator.cc:1051] Bin (512): 	Total Chunks: 11, Chunks in use: 11. 5.8KiB allocated for chunks. 5.8KiB in use in bin. 5.3KiB client-requested in use in bin.
2024-08-14 03:58:01.139774: I ex

ResourceExhaustedError: {{function_node __wrapped__AddV2_device_/job:localhost/replica:0/task:0/device:GPU:0}} failed to allocate memory [Op:AddV2] name: 